# Contradiction check (delta debugging)

Runs the delta-debugging contradiction check of `SHWClassifier` on a few tasks.

Every task in a `*_contradictory` dataset contains a contradiction, so the detection
rate there is the recall. The control section checks the same tasks on the original
dataset, where detections are false positives.

In [1]:
import json
import os
import sys
import time

# Run from the repository root so that relative dataset paths and `clarify` imports work
REPO_ROOT = (
    os.path.abspath("..") if os.path.basename(os.getcwd()) == "scripts" else os.getcwd()
)
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

from clarify.algorithms.shw_classifier import ContradictionDetector, decompose_prompt
from clarify.env import ClarificationConfiguration, ClarificationEnvironment

/Users/sihyun/Documents/kaist/coinse/clarification-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Parameters

In [2]:
DATASET_PATH = "data/humaneval/humaneval_larbi_contradictory.jsonl"
CONTROL_DATASET = "data/humaneval/humaneval_original.jsonl"

N = 5  # Number of tasks from the start of the dataset (ignored with TASK_IDS)
TASK_IDS = None  # e.g. {"HumanEval/2", "HumanEval/3"}
DECOMPOSITION = "heuristic"  # "heuristic" or "llm"
LANGUAGE_MODEL = "openai/gpt-4.1-mini"
DRY_RUN = False  # Only print the heuristic decomposition (no LLM or Docker calls)
OUTPUT_PATH = None  # Optional JSONL file to store the results

config = ClarificationConfiguration(language_model=LANGUAGE_MODEL)

## Helpers

In [3]:
def load(path, task_ids, n):
    with open(path, "r") as lines:
        rows = [json.loads(line) for line in lines if line.strip()]
    rows = [row for row in rows if row.get("prompt")]
    if task_ids:
        return [row for row in rows if row["task_id"] in task_ids]
    return rows[:n]


def check(row, config, decomposition, dry_run):
    problem = {"prompt": row["prompt"], "entry_point": row["entry_point"]}
    result = {"task_id": row["task_id"]}

    if decomposition == "heuristic" or dry_run:
        _, elements = decompose_prompt(problem, "heuristic")
        result["elements"] = elements
    if dry_run:
        return result

    num_llm_calls = 0

    def count_llm_calls(message):
        nonlocal num_llm_calls
        num_llm_calls += message["role"] == "assistant"

    env = ClarificationEnvironment(config, row, llm_api_hook=count_llm_calls)
    detector = ContradictionDetector(problem, decomposition)

    start = time.time()
    try:
        detected, description = detector.delta_debugging(env)
        result.update(detected=detected, description=description)
    except Exception as e:
        result.update(detected=None, description=f"[EXCEPTION] {type(e).__name__}: {e}")

    result.update(
        seconds=round(time.time() - start, 1),
        llm_calls=num_llm_calls,
        prompt_cost=round(env.prompt_cost, 6),
    )
    return result


def print_result(result):
    print(f"\n===== {result['task_id']}")
    for i, element in enumerate(result.get("elements", [])):
        print(f"  [{i}] {element!r}")
    if "detected" not in result:
        return
    print(
        f"  detected={result['detected']}  llm_calls={result['llm_calls']}"
        f"  cost=${result['prompt_cost']:.4f}  time={result['seconds']}s"
    )
    if result["description"]:
        for line in result["description"].splitlines():
            print(f"    {line}")


def summarize(name, results):
    finished = [r for r in results if r.get("detected") is not None]
    detected = sum(r["detected"] for r in finished)
    cost = sum(r["prompt_cost"] for r in results)
    print(
        f"\n{name}: detected {detected}/{len(finished)}  (errors: {len(results) - len(finished)})"
    )
    print(f"  total cost=${cost:.4f}")


def run(name, path, task_ids):
    rows = load(path, task_ids, N)
    print(f"########## {name}: {path} ({len(rows)} tasks)")
    results = []
    for row in rows:
        result = check(row, config, DECOMPOSITION, DRY_RUN)
        print_result(result)
        results.append(result)
    if not DRY_RUN:
        summarize(name, results)
    return [{"dataset": path, **r} for r in results]

## Contradictory dataset

In [4]:
results = run("contradictory", DATASET_PATH, TASK_IDS)

########## contradictory: data/humaneval/humaneval_larbi_contradictory.jsonl (5 tasks)

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


===== HumanEval/0
  [0] 'Check if in given list of numbers, are any two numbers closer to each other than given threshold.'
  [1] 'The function should also sort the list in ascending order'
  [2] 'but maintain the original order.'
  [3] 'The function must return an integer type result.'
  [4] 'The f

## Control: original dataset

Skip this cell if no control run is needed. It uses the same tasks as above.

In [5]:
control_task_ids = {r["task_id"] for r in results}
control_results = run("original (control)", CONTROL_DATASET, control_task_ids)

########## original (control): data/humaneval/humaneval_original.jsonl (5 tasks)

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


===== HumanEval/0
  [0] 'Check if in given list of numbers, are any two numbers closer to each other than given threshold.'
  [1] '>>> has_close_elements([1.0, 2.0, 3.0], 0.5)\nFalse'
  [2] '>>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)\nTrue'
  detected=None  llm_calls=0  cost=$0.0000  time

## Results table

In [6]:
all_results = results + globals().get("control_results", [])

print(
    f"{'dataset':<55} {'task_id':<16} {'detected':<9} {'llm_calls':>9} {'cost':>9} {'time':>7}"
)
for r in all_results:
    print(
        f"{r['dataset']:<55} {r['task_id']:<16} {str(r.get('detected')):<9}"
        f" {r.get('llm_calls', ''):>9} {r.get('prompt_cost', 0):>9.4f} {r.get('seconds', ''):>7}"
    )

dataset                                                 task_id          detected  llm_calls      cost    time
data/humaneval/humaneval_larbi_contradictory.jsonl      HumanEval/0      None              0    0.0000     1.9
data/humaneval/humaneval_larbi_contradictory.jsonl      HumanEval/1      None              0    0.0000     0.0
data/humaneval/humaneval_larbi_contradictory.jsonl      HumanEval/2      None              0    0.0000     0.0
data/humaneval/humaneval_larbi_contradictory.jsonl      HumanEval/3      None              0    0.0000     0.0
data/humaneval/humaneval_larbi_contradictory.jsonl      HumanEval/4      None              0    0.0000     0.0
data/humaneval/humaneval_original.jsonl                 HumanEval/0      None              0    0.0000     0.0
data/humaneval/humaneval_original.jsonl                 HumanEval/1      None              0    0.0000     0.0
data/humaneval/humaneval_original.jsonl                 HumanEval/2      None              0    0.0000     0.0
d

## Save

In [7]:
if OUTPUT_PATH:
    with open(OUTPUT_PATH, "w") as f:
        for result in all_results:
            f.write(json.dumps(result) + "\n")
    print(f"Saved results to {OUTPUT_PATH}")